# player_stance 3-class 분류기 학습 (Google Colab)

lotte-insight 프로젝트 — `training/train_player_stance_classifier.py` Colab 실행용 노트북

**모델:** `monologg/koelectra-small-v3-discriminator` fine-tuning  
**분류 방식:** 3-class CrossEntropyLoss (negative / neutral / positive)  
**학습 데이터:** `labeled_players.csv`의 `player_stance` 컬럼 (~ 4,465건 예상)  
**입력:** title (seq-A) + query_player + description_snippet (seq-B)  
**목표:** val macro F1 ≥ 0.70  
**예상 소요:** T4 GPU 기준 약 5~8분

**사전 준비**
- 런타임 유형: T4 GPU (런타임 → 런타임 유형 변경)
- 업로드할 파일: `training/data/labeled_players.csv`

In [ ]:
# 1. GPU 확인
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA:', torch.version.cuda)
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('[WARN] GPU 없음 — CPU 학습 시 약 2~3시간 소요')

In [ ]:
# 2. 레포 클론 및 의존성 설치
GITHUB_REPO_URL = 'https://github.com/JoeYunHa/Lotte_Insight.git'

!git clone {GITHUB_REPO_URL} /content/lotte-insight
%cd /content/lotte-insight/training
!pip install -q transformers torch scikit-learn pandas numpy

In [ ]:
# 3. 학습 데이터 업로드
import os
from google.colab import files

DATA_DIR = '/content/lotte-insight/training/data'
os.makedirs(DATA_DIR, exist_ok=True)

uploaded = files.upload()

for fname, content in uploaded.items():
    dst = f'{DATA_DIR}/{os.path.basename(fname)}'
    with open(dst, 'wb') as f:
        f.write(content)
    print(f'저장 완료: {dst}  ({len(content):,} bytes)')

In [ ]:
# 4. 학습 데이터 분포 확인
import pandas as pd

PLAYER_STANCE_LABELS = ['negative', 'neutral', 'positive']

path = f'{DATA_DIR}/labeled_players.csv'
if not os.path.exists(path):
    raise FileNotFoundError('[MISSING] labeled_players.csv')

df = pd.read_csv(path, encoding='utf-8-sig')
df_labeled = df.dropna(subset=['player_stance'])
df_labeled = df_labeled[df_labeled['player_stance'].isin(PLAYER_STANCE_LABELS)]

print(f'labeled_players.csv: 총 {len(df)}행 → 학습 가능 {len(df_labeled)}행 (제외 {len(df) - len(df_labeled)})')
for label in PLAYER_STANCE_LABELS:
    cnt = (df_labeled['player_stance'] == label).sum()
    print(f'  {label}: {cnt}')

print(f'\nquery_player 분포 (상위 20):')
print(df_labeled['query_player'].value_counts().head(20))

In [ ]:
# 5. 학습 실행 (T4 GPU 기준 약 5~8분)
!python train_player_stance_classifier.py \
    --data-dir /content/lotte-insight/training/data \
    --output-dir /content/lotte-insight/training/models/player_stance_koelectra \
    --epochs 5 \
    --lr 5e-5 \
    --batch 16

In [ ]:
# 6. 학습 결과 확인
import json, os

MODEL_DIR = '/content/lotte-insight/training/models/player_stance_koelectra'
config_path = f'{MODEL_DIR}/player_stance_config.json'

if os.path.exists(config_path):
    with open(config_path) as f:
        cfg = json.load(f)
    print('저장된 라벨 매핑:', cfg['label2id'])
    print('배포 시 PLAYER_STANCE_CLASSIFIER_MODEL_DIR=/app/models/player_stance_koelectra 설정 필요')
else:
    print('[ERROR] player_stance_config.json 없음 — 학습 실패 여부 확인')

print('\n모델 파일 목록:')
for fn in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(f'{MODEL_DIR}/{fn}')
    print(f'  {fn:<40} {size:>10,} bytes')

In [ ]:
# 7. Smoke test
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = '/content/lotte-insight/training/models/player_stance_koelectra'
LABELS = ['negative', 'neutral', 'positive']

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

def predict(title, player, snippet=''):
    player_snippet = f'{player} {snippet[:300].strip()}'.strip()
    enc = tokenizer(
        title, player_snippet,
        truncation='only_second', padding='max_length',
        max_length=128, return_tensors='pt',
    )
    with torch.no_grad():
        logits = model(**enc).logits[0]
        probs = torch.softmax(logits, dim=-1).tolist()
    best = max(range(len(probs)), key=lambda i: probs[i])
    return {'label': LABELS[best], 'confidence': round(probs[best], 4)}

# (title, player, snippet, expected)
test_cases = [
    ('나균안, 7이닝 무실점 완벽 투구', '나균안', '나균안이 선발 등판해 7이닝 무실점으로 팀 승리를 이끌었다.', 'positive'),
    ('롯데 불펜진 붕괴…박세웅 9실점 대패', '박세웅', '박세웅이 9실점을 허용하며 팀 패배의 원인이 됐다.', 'negative'),
    ('전준우, 내일 삼성전 4번 타자 선발 예고', '전준우', '롯데는 내일 전준우를 4번 타자로 기용할 예정이다.', 'neutral'),
    ('롯데 자이언츠, 삼성 7대3 대승', '이민석', '이민석이 결승 홈런을 포함해 3타점을 기록했다.', 'positive'),
    ('롯데 외국인 투수 방출…나균안도 부상 이탈', '나균안', '나균안이 부상으로 1군 엔트리에서 말소됐다.', 'negative'),
]

print('=== Smoke Test ===')
passed = 0
for title, player, snippet, expected in test_cases:
    result = predict(title, player, snippet)
    ok = result['label'] == expected
    mark = 'O' if ok else 'X'
    if ok:
        passed += 1
    print(f'{mark} [{result["label"]:>8}] conf={result["confidence"]:.3f}  (기대: {expected})  [{player}]')
    print(f'   {title}')
    print()

print(f'결과: {passed}/{len(test_cases)} 통과')

In [ ]:
# 8. 모델 다운로드 (Colab → 로컬)
import shutil, zipfile
from google.colab import files

MODEL_DIR = '/content/lotte-insight/training/models/player_stance_koelectra'
ZIP_PATH = '/content/player_stance_koelectra.zip'

shutil.make_archive('/content/player_stance_koelectra', 'zip', MODEL_DIR)
print(f'압축 완료: {ZIP_PATH}')
with zipfile.ZipFile(ZIP_PATH) as z:
    for name in sorted(z.namelist()):
        info = z.getinfo(name)
        print(f'  {name:<45} {info.file_size:>10,} bytes')
files.download(ZIP_PATH)

## 다운로드 후 로컬 배치

```
player_stance_koelectra.zip 압축 해제
  → training/models/player_stance_koelectra/
```

필수 파일:
- `config.json`, `pytorch_model.bin` (또는 `model.safetensors`)
- `tokenizer_config.json`, `vocab.txt`
- `player_stance_config.json` ← 라벨 매핑

배치 완료 후 `.env`에 추가:
```
PLAYER_STANCE_CLASSIFIER_MODEL_DIR=/app/models/player_stance_koelectra
```

`backend/models/player_stance_classifier.py`의 `classify_player_stance(title, description_snippet, player_name)` 또는
`classify_player_stance_batch(articles)` — articles에 `player_name` 필드 포함.